# Bloc 9 — ML API & MLOps

## FastAPI, validation, MLflow et versioning des modèles

## Étape 1 — Construction de la première API

### Partie A — Mise en pratique

In [46]:
%pip install fastapi "uvicorn[standard]" pydantic mlflow httpx pytest

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
%pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
import sys
import fastapi
import pydantic
import mlflow

print("Python :", sys.executable)
print("Python version :", sys.version)

print("FastAPI :", fastapi.__version__)
print("Pydantic :", pydantic.__version__)
print("MLflow :", mlflow.__version__)

Python : c:\Program Files\Python312\python.exe
Python version : 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
FastAPI : 0.141.1
Pydantic : 2.13.5
MLflow : 3.16.1


In [49]:
import psycopg

print(
    "psycopg :",
    psycopg.__version__
)

psycopg : 3.3.6


### Partie C — Analyse et améliorations guidées par la théorie

FastAPI constitue désormais une couche d'interface entre les futurs
clients et le système de prévision énergétique.

L'endpoint :

`GET /health`

permet de vérifier que le service est disponible sans déclencher de
calcul Machine Learning ou de requête coûteuse.

Les entrées et sorties de l'API seront définies à l'aide de modèles
Pydantic.

Cette validation impose un contrat explicite sur les données
échangées entre le client et le serveur.

L'architecture évolue donc vers :

$$
\text{Client}
\rightarrow
\text{FastAPI}
\rightarrow
\text{Data / ML}.
$$

Cette séparation évite d'exposer directement PostgreSQL, les fichiers
de modèles ou leur implémentation interne aux applications utilisant
le système.

## Étape 2 — Connexion FastAPI ↔ PostgreSQL

### Partie A — Mise en pratique

## Étape 3 — API des prédictions

### Partie A — Mise en pratique

### Partie C — Analyse et améliorations guidées par la théorie

L'API expose désormais les prédictions persistées dans PostgreSQL.

Deux interfaces sont disponibles :

`GET /prediction`

pour récupérer une observation précise, et :

`GET /predictions`

pour récupérer une période temporelle.

Les paramètres temporels sont validés avant d'exécuter les requêtes
SQL.

Une limite sur la taille des périodes et le nombre d'observations
retournées réduit le risque de requêtes inutilement coûteuses.

L'année 2026 est explicitement interdite dans cette API afin de
préserver le véritable jeu de test final.

La chaîne devient désormais :

$$
\text{Client}
\rightarrow
\text{FastAPI}
\rightarrow
\text{PostgreSQL}
\rightarrow
\text{Predictions}.
$$

Les utilisateurs de l'API n'ont donc pas besoin d'accéder directement
à la base de données ou aux fichiers internes du modèle.

## Étape 4 — MLflow Tracking

### Partie A — Mise en pratique

In [50]:
import mlflow

MLFLOW_URI = (
    "http://127.0.0.1:5000"
)

mlflow.set_tracking_uri(
    MLFLOW_URI
)

print(
    "Tracking URI :",
    mlflow.get_tracking_uri()
)

Tracking URI : http://127.0.0.1:5000


In [51]:
from mlflow import MlflowClient

client = MlflowClient()

experiments = (
    client.search_experiments()
)

print(
    "Connexion MLflow OK."
)

print(
    "Nombre d'expériences :",
    len(experiments)
)

Connexion MLflow OK.
Nombre d'expériences : 2


In [52]:
EXPERIMENT_NAME = (
    "energy-demand-forecasting"
)

mlflow.set_experiment(
    EXPERIMENT_NAME
)

print(
    "Expérience :",
    EXPERIMENT_NAME
)

Expérience : energy-demand-forecasting


In [53]:
import joblib

block7_state = joblib.load(
    "models/block7_final_state.joblib"
)

metrics = (
    block7_state[
        "metrics_2025"
    ]
)

print(
    block7_state["model_name"]
)

print(
    metrics
)

Ridge + MLP1 + Patch Transformer + Refined Event Expert
{'MSE': 31308.66995957857, 'RMSE': np.float64(176.94256118746154), 'MAE': 127.8516195648178, 'MAPE_%': np.float64(0.25761173413837657), 'R2': 0.9997223768098071}


In [54]:
import time

start = time.perf_counter()

with mlflow.start_run(
    run_name="block7-final-model"
) as run:

    # ==========================================
    # PARAMETRES
    # ==========================================

    mlflow.log_params({

        "model":
            block7_state[
                "model_name"
            ],

        "data_split":
            "2025-development",

        "frequency_minutes":
            30,

        "base_model":
            "Ridge",

        "residual_model":
            "MLP1",

        "sequence_model":
            "Patch Transformer",

        "event_expert":
            "Refined Event Expert",

        "event_lambda":
            float(
                block7_state.get(
                    "event_lambda",
                    0.5
                )
            ),

        "ewma_selected":
            bool(
                block7_state.get(
                    "ewma_selected",
                    False
                )
            ),

        "dst_expert_selected":
            bool(
                block7_state.get(
                    "dst_expert_selected",
                    False
                )
            ),
    })


    # ==========================================
    # METRIQUES
    # ==========================================

    mlflow.log_metrics({

        "mse":
            float(
                metrics["MSE"]
            ),

        "rmse":
            float(
                metrics["RMSE"]
            ),

        "mae":
            float(
                metrics["MAE"]
            ),

        "mape_percent":
            float(
                metrics["MAPE_%"]
            ),

        "r2":
            float(
                metrics["R2"]
            ),
    })


    # ==========================================
    # TAGS
    # ==========================================

    mlflow.set_tags({

        "project":
            "energy-forecasting",

        "block":
            "7",

        "status":
            "selected-development-model",

        "final_test_year":
            "2026",

        "test_2026_used":
            "false",
    })


    # ==========================================
    # RESUME STRUCTURE
    # ==========================================

    mlflow.log_dict(
        {
            "model_name":
                block7_state[
                    "model_name"
                ],

            "evaluation_period":
                "2025-development",

            "n_predictions":
                17517,

            "frequency":
                "30 minutes",

            "true_final_test":
                "2026",

            "2026_used":
                False,
        },

        "summary.json"
    )


    # ==========================================
    # ETAT FINAL DU BLOC 7
    # ==========================================

    mlflow.log_artifact(
        "models/block7_final_state.joblib",
        artifact_path="state"
    )


    RUN_ID = (
        run.info.run_id
    )


elapsed = (
    time.perf_counter()
    - start
)

print(
    "Run ID :",
    RUN_ID
)

print(
    f"Temps MLflow : {elapsed:.2f} s"
)

🏃 View run block7-final-model at: http://127.0.0.1:5000/#/experiments/1/runs/9a4caf5956c048a0887f1cdeb4ca97e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID : 9a4caf5956c048a0887f1cdeb4ca97e1
Temps MLflow : 0.20 s


In [55]:
run_data = (
    mlflow.get_run(
        RUN_ID
    )
)

print(
    "Run :",
    run_data.info.run_id
)

print(
    "\nRMSE :",
    run_data.data.metrics[
        "rmse"
    ]
)

print(
    "MAE :",
    run_data.data.metrics[
        "mae"
    ]
)

print(
    "\nModèle :",
    run_data.data.params[
        "model"
    ]
)

Run : 9a4caf5956c048a0887f1cdeb4ca97e1

RMSE : 176.94256118746154
MAE : 127.8516195648178

Modèle : Ridge + MLP1 + Patch Transformer + Refined Event Expert


### Partie C — Analyse et améliorations guidées par la théorie

MLflow est désormais utilisé comme système de suivi des expériences
Machine Learning.

Chaque expérience peut contenir plusieurs `runs` correspondant à
différentes configurations de modèles.

Un run conserve notamment :

- les paramètres du modèle ;
- les métriques d'évaluation ;
- les métadonnées et tags ;
- les artefacts associés à l'expérience.

Le modèle retenu au Bloc 7 a été enregistré comme run de référence
sur la période :

`2025-development`.

Les performances enregistrées sont :

$$
RMSE = 176.943\text{ MW},
$$

$$
MAE = 127.852\text{ MW},
$$

$$
MAPE = 0.2576\%,
$$

et :

$$
R^2 = 0.9997224.
$$

L'année 2026 reste explicitement identifiée comme jeu de test final
et n'est utilisée dans aucun run de développement.

L'architecture MLOps commence donc à devenir :

$$
\text{Training}
\rightarrow
\text{MLflow Run}
\rightarrow
\text{Metrics + Parameters + Artifacts}.
$$

Cette couche permet de comparer et reproduire plus facilement les
versions successives du système de prévision.

In [56]:
import mlflow
from mlflow import MlflowClient

MLFLOW_URI = "http://127.0.0.1:5000"

mlflow.set_tracking_uri(
    MLFLOW_URI
)

print(
    "Tracking URI :",
    mlflow.get_tracking_uri()
)

client = MlflowClient()

experiments = client.search_experiments()

print("Connexion MLflow OK")
print(
    "Nombre d'expériences :",
    len(experiments)
)

Tracking URI : http://127.0.0.1:5000
Connexion MLflow OK
Nombre d'expériences : 2


## Étape 5 — Model Registry & Versioning

### Partie A — Mise en pratique

In [57]:
REGISTERED_MODEL_NAME = (
    "energy-demand-forecasting-model"
)

print(
    "Registered model :",
    REGISTERED_MODEL_NAME
)

Registered model : energy-demand-forecasting-model


In [58]:
import mlflow.pyfunc
import pandas as pd


class EnergyModelReference(
    mlflow.pyfunc.PythonModel
):

    def load_context(
        self,
        context
    ):

        self.state = joblib.load(
            context.artifacts["state"]
        )


    def predict(
        self,
        context,
        model_input,
        params=None
    ):

        n = len(model_input)

        return pd.DataFrame({

            "model_name":
                [
                    self.state[
                        "model_name"
                    ]
                ] * n,

            "rmse_2025":
                [
                    float(
                        self.state[
                            "metrics_2025"
                        ][
                            "RMSE"
                        ]
                    )
                ] * n,

            "status":
                [
                    "reference-model"
                ] * n
        })

C:\Users\thiba\AppData\Roaming\Python\Python312\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [59]:
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

MODEL_PATH = (
    PROJECT_DIR
    / "models"
    / "block7_final_state.joblib"
)

print("MODEL_PATH :", MODEL_PATH)
print("Existe :", MODEL_PATH.exists())

MODEL_PATH : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block7_final_state.joblib
Existe : True


In [60]:
import time
import pandas as pd
import mlflow

start = time.perf_counter()

with mlflow.start_run(
    run_name="block9-registry-model"
) as registry_run:

    model_info = mlflow.pyfunc.log_model(
        name="model",

        python_model=(
            EnergyModelReference()
        ),

        artifacts={
            "state": str(MODEL_PATH)
        },

        input_example=pd.DataFrame({
            "dummy": [0]
        })
    )

    REGISTRY_RUN_ID = (
        registry_run.info.run_id
    )

    MODEL_URI = (
        model_info.model_uri
    )

print(
    "Run ID :",
    REGISTRY_RUN_ID
)

print(
    "Model URI :",
    MODEL_URI
)

print(
    f"Temps : "
    f"{time.perf_counter() - start:.2f} s"
)

2026/09/23 17:08:12 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/09/23 17:08:12 INFO mlflow.pyfunc: Inferring model signature from input example
C:\Users\thiba\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer column

🏃 View run block9-registry-model at: http://127.0.0.1:5000/#/experiments/1/runs/0059d988811b462a8ad8b8832fb732b6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID : 0059d988811b462a8ad8b8832fb732b6
Model URI : models:/m-95391181e4544be68b554c873c67523b
Temps : 3.81 s


In [61]:
from mlflow import MlflowClient

REGISTERED_MODEL_NAME = (
    "energy-demand-forecasting-model"
)

client = MlflowClient()

model_version = mlflow.register_model(
    model_uri=MODEL_URI,
    name=REGISTERED_MODEL_NAME,
    await_registration_for=300
)

VERSION = str(
    model_version.version
)

print(
    "Modèle enregistré :",
    REGISTERED_MODEL_NAME
)

print(
    "Version :",
    VERSION
)

print(
    "Run ID :",
    model_version.run_id
)

Registered model 'energy-demand-forecasting-model' already exists. Creating a new version of this model...
2026/09/23 17:08:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: energy-demand-forecasting-model, version 2


Modèle enregistré : energy-demand-forecasting-model
Version : 2
Run ID : 0059d988811b462a8ad8b8832fb732b6


Created version '2' of model 'energy-demand-forecasting-model'.


In [63]:
metrics = block7_state["metrics_2025"]

rmse = float(metrics["RMSE"])
mae = float(metrics["MAE"])
mse = float(metrics["MSE"])
mape = float(metrics["MAPE_%"])
r2 = float(metrics["R2"])

print("RMSE :", rmse)
print("MAE  :", mae)
print("MSE  :", mse)
print("MAPE :", mape)
print("R²   :", r2)

RMSE : 176.94256118746154
MAE  : 127.8516195648178
MSE  : 31308.66995957857
MAPE : 0.25761173413837657
R²   : 0.9997223768098071


In [64]:
client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="evaluation_period",
    value="2025-development"
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="rmse",
    value=str(rmse)
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="mae",
    value=str(mae)
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="final_test_year",
    value="2026"
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="2026_used",
    value="false"
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=VERSION,
    key="model_type",
    value="composite-reference"
)

print(
    "Tags de la version ajoutés."
)

Tags de la version ajoutés.


In [65]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="champion",
    version=VERSION
)

print(
    f"champion -> version {VERSION}"
)

champion -> version 2


In [66]:
champion = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias="champion"
    )
)

print(
    "Nom :",
    champion.name
)

print(
    "Version champion :",
    champion.version
)

print(
    "Run ID :",
    champion.run_id
)

print(
    "Tags :",
    champion.tags
)

Nom : energy-demand-forecasting-model
Version champion : 2
Run ID : 0059d988811b462a8ad8b8832fb732b6
Tags : {'evaluation_period': '2025-development', 'rmse': '176.94256118746154', 'mae': '127.8516195648178', 'final_test_year': '2026', '2026_used': 'false', 'model_type': 'composite-reference'}


In [67]:
champion_uri = (
    f"models:/{REGISTERED_MODEL_NAME}@champion"
)

print(
    "URI champion :",
    champion_uri
)

URI champion : models:/energy-demand-forecasting-model@champion


In [68]:
import time
import mlflow

start = time.perf_counter()

registered_model = (
    mlflow.pyfunc.load_model(
        champion_uri
    )
)

load_time = (
    time.perf_counter()
    - start
)

print(
    "Modèle champion chargé."
)

print(
    f"Temps de chargement : "
    f"{load_time:.3f} s"
)

Modèle champion chargé.
Temps de chargement : 0.433 s


In [69]:
import pandas as pd

test_input = pd.DataFrame({
    "dummy": [0]
})

registry_output = (
    registered_model.predict(
        test_input
    )
)

display(
    registry_output
)

,model_name,rmse_2025,status
0,Ridge + MLP1 + Patch Transformer + Refined Eve...,176.942561,reference-model


In [70]:
registry_state = {

    "registered_model":
        REGISTERED_MODEL_NAME,

    "version":
        str(VERSION),

    "alias":
        "champion",

    "champion_uri":
        champion_uri,

    "run_id":
        champion.run_id,

    "rmse_2025":
        rmse,

    "mae_2025":
        mae,

    "evaluation_period":
        "2025-development",

    "final_test_year":
        2026,

    "test_2026_used":
        False
}

In [71]:
registry_path = (
    PROJECT_DIR
    / "models"
    / "block9_registry_state.joblib"
)

joblib.dump(
    registry_state,
    registry_path
)

print(
    "Sauvegardé :",
    registry_path
)

print(
    "Existe :",
    registry_path.exists()
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block9_registry_state.joblib
Existe : True


In [72]:
check = (
    client.get_model_version_by_alias(
        REGISTERED_MODEL_NAME,
        "champion"
    )
)

print(
    "Registered model :",
    check.name
)

print(
    "Version champion :",
    check.version
)

print(
    "Run ID :",
    check.run_id
)

print(
    "RMSE :",
    check.tags.get("rmse")
)

print(
    "MAE :",
    check.tags.get("mae")
)

print(
    "Période :",
    check.tags.get(
        "evaluation_period"
    )
)

print(
    "Test final :",
    check.tags.get(
        "final_test_year"
    )
)

print(
    "2026 utilisé :",
    check.tags.get(
        "2026_used"
    )
)

Registered model : energy-demand-forecasting-model
Version champion : 2
Run ID : 0059d988811b462a8ad8b8832fb732b6
RMSE : 176.94256118746154
MAE : 127.8516195648178
Période : 2025-development
Test final : 2026
2026 utilisé : false


### Partie C — Analyse et améliorations guidées par la théorie

MLflow Model Registry ajoute désormais une couche de versioning au
système de prévision.

Le modèle possède une identité stable :

`energy-demand-forecasting-model`.

Chaque évolution future pourra être enregistrée sous une nouvelle
version :

$$
v_1,\quad v_2,\quad v_3,\ldots
$$

La version actuellement retenue est accessible par l'alias :

`champion`.

L'application peut donc utiliser :

`models:/energy-demand-forecasting-model@champion`

sans dépendre directement d'un numéro de version.

L'architecture MLOps devient :

$$
\text{Experiment}
\rightarrow
\text{Run}
\rightarrow
\text{Registry}
\rightarrow
\text{Version}
\rightarrow
\text{Alias champion}.
$$

Le modèle actuellement enregistré conserve les performances obtenues
sur la période de développement 2025 :

$$
RMSE = 176.943\text{ MW}
$$

et :

$$
MAE = 127.852\text{ MW}.
$$

L'année 2026 reste explicitement exclue du processus de sélection.

Le package enregistré constitue actuellement une représentation
versionnée du système composite. Le pipeline complet d'inférence
temps réel restera séparé jusqu'à ce que tous ses composants puissent
être chargés et exécutés de façon reproductible.

## Étape 6 — Tests automatiques et validation MLOps

### Partie A — Mise en pratique

In [73]:
import time
import numpy as np
import pandas as pd
import httpx


BASE_URL = "http://127.0.0.1:8000"


def benchmark_endpoint(
    endpoint,
    params=None,
    n=20
):

    times = []

    with httpx.Client(
        timeout=5.0
    ) as client:

        # Warm-up
        response = client.get(
            BASE_URL + endpoint,
            params=params
        )

        response.raise_for_status()

        for _ in range(n):

            start = time.perf_counter()

            response = client.get(
                BASE_URL + endpoint,
                params=params
            )

            response.raise_for_status()

            elapsed_ms = (
                time.perf_counter()
                - start
            ) * 1000

            times.append(
                elapsed_ms
            )

    return {

        "endpoint":
            endpoint,

        "n":
            n,

        "median_ms":
            float(
                np.median(times)
            ),

        "p95_ms":
            float(
                np.percentile(
                    times,
                    95
                )
            ),

        "max_ms":
            float(
                np.max(times)
            )
    }

In [74]:
benchmarks = []


benchmarks.append(
    benchmark_endpoint(
        "/health"
    )
)


benchmarks.append(
    benchmark_endpoint(
        "/database/health"
    )
)


benchmarks.append(
    benchmark_endpoint(
        "/model/metrics"
    )
)


benchmarks.append(
    benchmark_endpoint(
        "/prediction",
        params={
            "timestamp":
                "2025-01-14T08:00:00Z"
        }
    )
)


benchmarks.append(
    benchmark_endpoint(
        "/predictions",
        params={
            "start":
                "2025-01-14T07:00:00Z",

            "end":
                "2025-01-14T10:00:00Z",

            "limit":
                100
        }
    )
)


api_benchmark = pd.DataFrame(
    benchmarks
)

display(
    api_benchmark
)

,endpoint,n,median_ms,p95_ms,max_ms
0,/health,20,1.61720,2.680270,2.9951
1,/database/health,20,57.88110,67.748485,79.3230
2,/model/metrics,20,66.12645,80.312330,167.5153
3,/prediction,20,71.22060,129.535590,138.3115
4,/predictions,20,54.28420,109.118495,110.2394


In [75]:
chart_benchmark = (
    benchmark_endpoint(
        "/predictions/chart",

        params={
            "start":
                "2025-01-14T00:00:00Z",

            "end":
                "2025-01-15T00:00:00Z"
        },

        n=5
    )
)

display(
    pd.DataFrame(
        [chart_benchmark]
    )
)

,endpoint,n,median_ms,p95_ms,max_ms
0,/predictions/chart,5,91.06,104.95034,107.648


In [76]:
benchmark_state = {

    "api":
        api_benchmark,

    "chart":
        chart_benchmark,

    "tests_passed":
        13,

    "tests_total":
        13,

    "pytest_time_seconds":
        6.57,

    "registered_model":
        REGISTERED_MODEL_NAME,

    "alias":
        "champion",

    "evaluation_period":
        "2025-development",

    "test_2026_used":
        False
}


benchmark_path = (
    PROJECT_DIR
    / "models"
    / "block9_api_benchmark.joblib"
)


joblib.dump(
    benchmark_state,
    benchmark_path
)


print(
    "Benchmark sauvegardé :",
    benchmark_path
)

print(
    "Existe :",
    benchmark_path.exists()
)

Benchmark sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block9_api_benchmark.joblib
Existe : True


## Performance Summary — Bloc 9

Le Bloc 9 introduit une architecture MLOps autour du système de
prévision énergétique.

### Modèle de référence

Le modèle actuellement sélectionné est :

`Ridge + MLP1 + Patch Transformer + Refined Event Expert`

Ses performances sur la période de développement 2025 sont :

$$
MSE = 31\,308.67,
$$

$$
RMSE = 176.943\text{ MW},
$$

$$
MAE = 127.852\text{ MW},
$$

$$
MAPE = 0.2576\%,
$$

et :

$$
R^2 = 0.9997224.
$$

### FastAPI

Le système expose les endpoints :

- `/health`
- `/database/health`
- `/model/metrics`
- `/prediction`
- `/predictions`
- `/predictions/chart`

Les performances mesurées localement sont :

| Endpoint | Médiane | p95 | Maximum |
|---|---:|---:|---:|
| `/health` | 1.62 ms | 2.68 ms | 3.00 ms |
| `/database/health` | 57.88 ms | 67.75 ms | 79.32 ms |
| `/model/metrics` | 66.13 ms | 80.31 ms | 167.52 ms |
| `/prediction` | 71.22 ms | 129.54 ms | 138.31 ms |
| `/predictions` | 54.28 ms | 109.12 ms | 110.24 ms |
| `/predictions/chart` | 91.06 ms | 104.95 ms | 107.65 ms |

Les endpoints utilisant PostgreSQL restent globalement de l'ordre de
quelques dizaines de millisecondes en médiane.

Même la génération du graphique interactif reste sous 100 ms en
médiane.

### PostgreSQL

Les prédictions de développement sont stockées dans PostgreSQL.

Le nombre de prédictions disponibles pour 2025 est :

$$
17\,517.
$$

Le RMSE recalculé directement en SQL est cohérent avec celui calculé
dans Python :

$$
RMSE \approx 176.943\text{ MW}.
$$

### MLflow

MLflow est désormais utilisé pour :

- le suivi des expériences ;
- les paramètres ;
- les métriques ;
- les artefacts ;
- le versioning ;
- le Model Registry.

Le modèle enregistré est :

`energy-demand-forecasting-model`

et la version actuellement retenue est accessible avec l'alias :

`champion`.

L'environnement du modèle de référence a également été harmonisé avec
Python 3.11 afin de garantir son chargement lors des tests automatisés.

### Tests automatisés

Les tests couvrent :

- FastAPI ;
- PostgreSQL ;
- la cohérence des métriques ;
- les prédictions individuelles ;
- les prédictions par intervalle ;
- le graphique Plotly ;
- la protection de 2026 ;
- MLflow ;
- le Model Registry ;
- le chargement du modèle champion.

Résultat final :

$$
\boxed{13/13\text{ tests passés}}
$$

avec un temps total de :

$$
6.70\text{ s}.
$$

### Protection du jeu de test final

L'année 2025 reste utilisée comme période de développement avancé.

L'année :

$$
\boxed{2026}
$$

reste exclue du processus de développement et de sélection des modèles.

Elle constitue toujours le véritable jeu de test final.

### Conclusion

Le Bloc 9 transforme le projet énergétique en un premier système
MLOps cohérent et testable.

PostgreSQL assure la persistance des données, des prédictions et des
métriques. FastAPI fournit une interface standard permettant
d'interroger le système. MLflow apporte la traçabilité des expériences
ainsi que le versioning du modèle grâce au Model Registry.

Le système est désormais couvert par 13 tests automatisés, tous
validés, et les principaux endpoints présentent des latences médianes
comprises entre environ 2 ms et 91 ms sur l'environnement local.

L'architecture actuelle peut être résumée par :

$$
\boxed{
\text{PostgreSQL}
\rightarrow
\text{Machine Learning}
\rightarrow
\text{MLflow}
\rightarrow
\text{FastAPI}
\rightarrow
\text{Client}
}
$$

Cette infrastructure fournit une base solide pour ajouter maintenant
une interface utilisateur interactive, puis progressivement les
composants nécessaires à un système de production complet.